<a href="https://colab.research.google.com/github/GodesAleksandra/DI-Bootcamp/blob/main/Exercises_MCP_LLM_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises (Student) - MCP Client with LLM

In [ ]:
!pip install -q mcp azure-ai-inference azure-core nest_asyncio requests

In [ ]:

import os
from pathlib import Path
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_LLM = False  # flip True if GITHUB_TOKEN is set


In [ ]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


In [ ]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoServer")

@mcp.tool()
def add(a: int, b: int) -> int:
    "Add two numbers."
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    'Multiply two integers.'
    return a * b

@mcp.tool()
def greet(name: str) -> str:
    "Return a greeting string."
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run()


## Exercise 1 (provide answer)

#To-Do: Why is STDIO transport simple for local MCP dev compared to HTTP?

Direct Comparison: STDIO vs. HTTPFeature
STDIO (Local Child Process)HTTP / SSE (Network Service)Lifecycle Management

The STDIO client starts the server process on launch and kills it on exit.In case of HTTP you must start the server separately, keep it running, and kill it when done.
STDIO Network Configuration - no ports to manage, no localhost conflicts, and no firewall rules to whitelist. HTTP - you must assign free ports (e.g., 8080) and manage endpoint paths.
STDIO Security & AuthInherited - the process runs with your local user permissions; no API keys or tokens are needed to connect. HTTP - requires setting up CORS policies or authentication tokens to prevent unauthorized local access.
STDIO Protocol FlowFull-Duplex - Pure JSON-RPC 2.0 streaming directly over stdin and stdout. HTTP - requires complex Server-Sent Events (SSE) mapping to allow the server to push updates back.
STDIO  latency - OS-level pipe communication without the overhead of network stack layers. HTTP - adds TCP handshake and HTTP header parsing overhead, even on a loopback address.

## Exercise 2

In [1]:

import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

async def ex2_connect():
    params = #To-Do: Create StdioServerParameters with command to run server.py
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()


SyntaxError: invalid syntax (3225060564.py, line 9)

In [ ]:
# In a new cell
await ex2_connect()
print("Exercise 2: OK (connected and initialized)")


## Exercise 3

In [ ]:
async def ex3_list():
    params = #To-Do: Create StdioServerParameters with command to run server.py
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            resources = #To-Do: list resources from session
            print("RESOURCES:", resources)
            tools = await session.list_tools()
            for t in tools.tools:
                print(t.name, t.inputSchema.get("properties", {}))


In [ ]:
await ex3_list()

RESOURCES: meta=None nextCursor=None resources=[]
add {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## Exercise 4

#To-Do: Explain how the conversion to llm tool happens in MCP server code ?

When you write an MCP server, the conversion of a standard programming function into an LLM tool happens through a process called Structured Metadata Extraction and Schema Generation.Instead of you manually writing complex configurations, the MCP server framework uses reflection or type definitions to automatically inspect your code at startup and translate it into a structured schema the LLM can understand.

In [ ]:

def convert_to_llm_tool(tool):
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }


## Exercise 5

**Plan & execute:** Use stub (or real) LLM to propose `tool_calls`, then execute them and print results for a prompt like “Add 2 to 20.”

In [ ]:

import asyncio
import json
import sys
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    if not use_real:
        return stub_plan(prompt, functions)
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))
    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "system", "content": "Plan MCP tool calls."},{"role": "user", "content": prompt}],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls


In [ ]:
async def ex5_run(prompt: str = "Add 2 to 20"):
    params = #To-Do: Create StdioServerParameters with command to run server.py
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            tools = await session.list_tools()
            functions = #To-Do: convert tools to llm tool format
            calls = #To-Do: get tool calls from call_llm
            print("tool_calls:", calls)
            for call in calls:
                result = await session.call_tool(call["name"], arguments=call["args"])
                print("result:", [getattr(c, "text", str(c)) for c in result.content])


In [ ]:
await ex5_run("Add 2 to 20")

tool_calls: [{'name': 'add', 'args': {'a': 2, 'b': 20}}]
result: ['22']


## Optional - add multiply(a, b) and rerun